In [2]:
import scipy.stats as stats

Spring CV

In [5]:
between = [35.45, 44.44, 41.5, 22.78]
within = [22.8, 33.01, 33.09, 12.25, 18.71, 30.98, 13.69, 36.58, 25.74]

stat, p = stats.levene(between, within, center='median')

print(stat, p)

0.021724740554297906 0.885488587420687


Summer CV


In [6]:
between = [23.43, 12.11, 21.41, 34.94]
within = [14.5, 8.24, 1.35, 4.53, 13.57, 12.92, 33.32, 15.16]

stat, p = stats.levene(between, within, center='median')

print(stat, p)

3.304027524434601e-05 0.9955267849051539


Spring Std

In [ ]:
between = [35.45, 44.44, 41.5, 22.78]
within = [22.8, 33.01, 33.09, 12.25, 18.71, 30.98, 13.69, 36.58, 25.74]

stat, p = stats.levene(between, within, center='median')

print(stat, p)

## Copilot magic

In [12]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "TrapID": ["T1-A","T1-B","T1-C","T1-D","T2-A","T2-B","T2-C","T2-D",
                "T3-A","T3-B","T3-C","T3-D","T5-A","T5-B","T5-C","T5-D",
                "T6-A","T6-B","T6-C","T6-D","T7-A","T7-B","T7-C","T7-D",
                "T8-A","T8-B","T8-C","T8-D"],
    "Type": ["Closed","Open","Open","Closed","Closed","Open","Open","Closed",
                "Closed","Open","Open","Closed","Closed","Open","Open","Closed",
                "Closed","Open","Open","Closed","Closed","Open","Open","Closed",
                "Closed","Open","Open","Closed"],
    "Spring": [47.01,27.53,14.51,29.55,47.49,71.69,np.nan,np.nan,
                np.nan,np.nan,47.31,38.78,54.75,31.81,24.15,27.57,
                63.38,62.84,29.18,31.87,21.82,25.20,14.88,17.06,
                24.00,np.nan,31.26,35.04],
    "Summer": [11.40,12.61,16.57,8.51,np.nan,15.14,np.nan,np.nan,
                np.nan,np.nan,np.nan,4.64,20.03,14.90,11.49,16.98,
                np.nan,17.65,8.83,15.06,12.78,13.11,9.67,12.44,
                12.44,np.nan,13.81,13.62]
})

# reshape to long format
df_long = df.melt(id_vars=["TrapID","Type"], 
                    value_vars=["Spring","Summer"],
                    var_name="Season", value_name="Weight").dropna()

# extract location (T1, T2, ...)
df_long["Location"] = df_long["TrapID"].str.extract(r"(T\d+)")
df_long["Trap"] = df_long["TrapID"].str.extract(r"-(.)")

spring = df_long[df_long["Season"] == "Spring"]
summer = df_long[df_long["Season"] == "Summer"]



In [13]:
import numpy as np

def variance_ratio_test(data, n_perm=5000):
    # Between-location variance
    between = data.groupby("Location")["Weight"].mean().var()

    # Within-location variance
    within = data.groupby("Location")["Weight"].var().mean()

    R_obs = between / within

    # Permutation test
    R_perm = []
    for _ in range(n_perm):
        shuffled = data.copy()
        shuffled["Location"] = np.random.permutation(shuffled["Location"])
        b = shuffled.groupby("Location")["Weight"].mean().var()
        w = shuffled.groupby("Location")["Weight"].var().mean()
        R_perm.append(b / w)

    p_value = np.mean(np.array(R_perm) >= R_obs)
    return R_obs, p_value


In [14]:
R_spring, p_spring = variance_ratio_test(spring)
print("Spring variance ratio:", R_spring)
print("Spring p-value:", p_spring)


Spring variance ratio: 1.102452476019367
Spring p-value: 0.0414


In [15]:
R_summer, p_summer = variance_ratio_test(summer)
print("Summer variance ratio:", R_summer)
print("Summer p-value:", p_summer)


Summer variance ratio: 1.446136004615188
Summer p-value: 0.0754
